<a href="https://colab.research.google.com/github/godara97/northstar-urban-mobility-analysis/blob/main/r_notebooks/03_sql_in_r.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# installing the packages i need for sql in r

install.packages("sqldf")
install.packages("ggplot2")
install.packages("dplyr")

cat("all packages installed!\n")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘gsubfn’, ‘proto’, ‘RSQLite’, ‘chron’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



all packages installed!


In [10]:
# loading libraries
library(sqldf)
library(ggplot2)
library(dplyr)

cat("libraries loaded!\n")

customers  <- read.csv('customers.csv')
orders     <- read.csv('orders.csv')
deliveries <- read.csv('deliveries.csv')
drivers    <- read.csv('drivers.csv')
vehicles   <- read.csv('vehicles.csv')
hubs       <- read.csv('hubs.csv')
incidents  <- read.csv('incidents.csv')
complaints <- read.csv('complaints.csv')
app_events <- read.csv('app_events.csv')

cat("all files loaded!\n")
cat("customers:", nrow(customers), "rows\n")
cat("orders:", nrow(orders), "rows\n")
cat("deliveries:", nrow(deliveries), "rows\n")
cat("drivers:", nrow(drivers), "rows\n")
cat("vehicles:", nrow(vehicles), "rows\n")
cat("incidents:", nrow(incidents), "rows\n")
cat("complaints:", nrow(complaints), "rows\n")
cat("app_events:", nrow(app_events), "rows\n")

cat("\nready!\n")

libraries loaded!
all files loaded!
customers: 650 rows
orders: 1250 rows
deliveries: 950 rows
drivers: 170 rows
vehicles: 120 rows
incidents: 280 rows
complaints: 320 rows
app_events: 640 rows

ready!


In [11]:
# quick data cleaning in R
fix_zone <- function(x) {
  x <- trimws(tolower(x))
  ifelse(x %in% c("airport", "airprt"), "Airport",
  ifelse(x %in% c("north", "nrth"), "North",
  ifelse(x %in% c("south", "sth"), "South",
  ifelse(x %in% c("central", "ctr", "ctrl"), "Central",
  ifelse(x %in% c("east"), "East",
  ifelse(x %in% c("west"), "West",
  ifelse(x %in% c("riverside", "riversd"), "Riverside",
  x)))))))
}

# apply to all zone columns
customers$home_zone    <- fix_zone(customers$home_zone)
orders$pickup_zone     <- fix_zone(orders$pickup_zone)
orders$dropoff_zone    <- fix_zone(orders$dropoff_zone)
drivers$base_zone      <- fix_zone(drivers$base_zone)
vehicles$assigned_zone <- fix_zone(vehicles$assigned_zone)

# fix missing values
customers$loyalty_score[is.na(customers$loyalty_score)] <- mean(customers$loyalty_score, na.rm=TRUE)
customers$preferred_channel[is.na(customers$preferred_channel)] <- "Unknown"
orders$booking_channel[is.na(orders$booking_channel)] <- "Unknown"
drivers$training_score[is.na(drivers$training_score)] <- mean(drivers$training_score, na.rm=TRUE)
vehicles$battery_health_pct[is.na(vehicles$battery_health_pct)] <- mean(vehicles$battery_health_pct, na.rm=TRUE)
complaints$compensation_amount[is.na(complaints$compensation_amount)] <- mean(complaints$compensation_amount, na.rm=TRUE)
incidents$resolved_hours[is.na(incidents$resolved_hours)] <- mean(incidents$resolved_hours, na.rm=TRUE)

cat("data cleaned and ready!\n")

data cleaned and ready!


## SQL Queries in R
Now I will use sqldf to write SQL queries on the dataframes.
This lets me ask business questions using SQL syntax.

In [13]:
# SQL Query 1
# checking overall delivery performance using sql

query1 <- sqldf("
  SELECT delivery_status,
         COUNT(*) AS total_deliveries,
         ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM deliveries), 1) AS percentage
  FROM deliveries
  GROUP BY delivery_status
  ORDER BY total_deliveries DESC
")

cat("Query 1 - Delivery Status Breakdown:\n")
print(query1)

Query 1 - Delivery Status Breakdown:
  delivery_status total_deliveries percentage
1          OnTime              616       64.8
2         Delayed              202       21.3
3          Failed              132       13.9


### Query 1 Finding:
64.8% of deliveries are on time but 35.2% have problems.
For a transport company 1 in 3 deliveries having issues
is a serious operational problem that needs addressing.

In [14]:
# SQL Query 2
# finding which zones have most failed deliveries

query2 <- sqldf("
  SELECT o.pickup_zone,
         COUNT(*) AS failed_deliveries
  FROM deliveries d
  JOIN orders o ON d.order_id = o.order_id
  WHERE d.delivery_status = 'Failed'
  GROUP BY o.pickup_zone
  ORDER BY failed_deliveries DESC
")

cat("Query 2 - Failed Deliveries by Zone:\n")
print(query2)

Query 2 - Failed Deliveries by Zone:
  pickup_zone failed_deliveries
1     Central                33
2       North                22
3        East                19
4   Riverside                18
5        West                14
6       South                14
7     Airport                12


### Query 2 Finding:
Central zone has the most delivery failures with 33 cases.
This is almost 3 times more than Airport zone which has 12.
Central zone needs urgent operational review.

In [15]:
# SQL Query 3
# what are customers complaining about most

query3 <- sqldf("
  SELECT complaint_type,
         COUNT(*) AS total_complaints,
         severity,
         ROUND(AVG(resolution_days), 1) AS avg_resolution_days
  FROM complaints
  GROUP BY complaint_type
  ORDER BY total_complaints DESC
")

cat("Query 3 - Complaint Types with Resolution Time:\n")
print(query3)

Query 3 - Complaint Types with Resolution Time:
     complaint_type total_complaints severity avg_resolution_days
1             Delay              101     High                 7.3
2      MissedPickup               64   Medium                 7.6
3          AppIssue               53     High                 8.6
4   DriverBehaviour               51   Medium                 8.2
5 SupportExperience               20   Medium                 7.5
6           Billing               16   Medium                 7.8
7            Damage               15      Low                11.3


### Query 3 Finding:
Delay is the top complaint with 101 cases.
Damage complaints take the longest to resolve at 11.3 days.
AppIssue complaints are High severity showing the mobile
app is a significant problem area for NorthStar.

In [16]:
# SQL Query 4
# finding drivers with most failed deliveries

query4 <- sqldf("
  SELECT d.driver_id,
         dr.base_zone,
         dr.driver_rating,
         COUNT(*) AS total_deliveries,
         SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed_count,
         ROUND(SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS failure_rate_pct
  FROM deliveries d
  JOIN drivers dr ON d.driver_id = dr.driver_id
  GROUP BY d.driver_id
  HAVING total_deliveries >= 5
  ORDER BY failure_rate_pct DESC
  LIMIT 10
")

cat("Query 4 - Top 10 Drivers with Highest Failure Rate:\n")
print(query4)

Query 4 - Top 10 Drivers with Highest Failure Rate:
   driver_id base_zone driver_rating total_deliveries failed_count
1       D092      East          4.24                5            3
2       D104      West          3.45                7            4
3       D024 Riverside          3.35                8            4
4       D010      West          3.95                7            3
5       D144      West          3.83                5            2
6       D143   Central          4.14                5            2
7       D095      West          3.15                5            2
8       D005     North          4.14                5            2
9       D165     North          3.89                6            2
10      D133     South          3.99               12            4
   failure_rate_pct
1              60.0
2              57.1
3              50.0
4              42.9
5              40.0
6              40.0
7              40.0
8              40.0
9              33.3
10         

### Query 4 Finding:
Driver D095 has the lowest rating of 3.15 and a 40% failure rate.
Most underperforming drivers are based in West and North zones.
These drivers need additional training or performance review.

In [17]:
# SQL Query 5
# checking if vehicles in repair cause more failures

query5 <- sqldf("
  SELECT v.maintenance_status,
         COUNT(*) AS total_deliveries,
         SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed_count,
         ROUND(SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS failure_rate_pct
  FROM deliveries d
  JOIN vehicles v ON d.vehicle_id = v.vehicle_id
  GROUP BY v.maintenance_status
  ORDER BY failure_rate_pct DESC
")

cat("Query 5 - Delivery Failures by Vehicle Maintenance Status:\n")
print(query5)

Query 5 - Delivery Failures by Vehicle Maintenance Status:
  maintenance_status total_deliveries failed_count failure_rate_pct
1           InRepair              254           77             30.3
2             Active              542           45              8.3
3          Scheduled              154           10              6.5


### Query 5 Finding:
This is the most important finding from SQL analysis.
Vehicles in repair have a 30.3% failure rate compared
to only 8.3% for active vehicles. This means NorthStar
is assigning deliveries to vehicles that are not fully
operational which is directly causing failures.

In [18]:
# SQL Query 6
# finding the most valuable customers by order value

query6 <- sqldf("
  SELECT c.customer_id,
         c.customer_type,
         c.home_zone,
         COUNT(o.order_id) AS total_orders,
         ROUND(SUM(o.order_value), 2) AS total_spent,
         ROUND(AVG(o.order_value), 2) AS avg_order_value
  FROM customers c
  JOIN orders o ON c.customer_id = o.customer_id
  GROUP BY c.customer_id
  ORDER BY total_spent DESC
  LIMIT 10
")

cat("Query 6 - Top 10 Most Valuable Customers:\n")
print(query6)

Query 6 - Top 10 Most Valuable Customers:
   customer_id customer_type home_zone total_orders total_spent avg_order_value
1        C0545      Consumer     South            6      923.45          153.91
2        C0157      Consumer   Central            4      720.04          180.01
3        C0622      Consumer Riverside            6      701.21          116.87
4        C0343      Consumer   Central            7      685.78           97.97
5        C0372      Consumer      West            6      669.11          111.52
6        C0013    Enterprise      East            5      659.26          131.85
7        C0558      Consumer     South            4      605.85          151.46
8        C0289           SME     North            4      601.81          150.45
9        C0301           SME Riverside            3      592.99          197.66
10       C0476      Consumer     North            5      587.86          117.57


### Query 6 Finding:
Consumer type customers generate the most revenue.
Top customer C0545 spent £923 across 6 orders.
NorthStar should prioritise service quality for
these high value customers to retain their business.